In [ ]:
"""
HOW TO USE THIS SCRIPT:

1. Save this file as:
   clean_labchart.py

2. Open a terminal or run it in Spyder / Jupyter.

3. Run the script:
   python clean_labchart.py

4. When prompted, enter the path to your LabChart file, for example:
   C:\\Users\\YourName\\Desktop\\ECG_data.txt

5. The script will automatically:
   - remove metadata
   - clean the data
   - convert commas to dots
   - create a folder on your Desktop called "LabChart_clean"

OUTPUT:
- ECG_YYYY-MM-DD_HH-MM-SS_kubios.txt   → ready for Kubios
- ECG_YYYY-MM-DD_HH-MM-SS_metadata.txt → original metadata
"""

In [7]:
import os
import re

# ---------------------------------------------------------
# Function: extract_datetime
# Purpose:
#   Extract recording date and time from LabChart metadata
#   and format it for file naming.
# ---------------------------------------------------------
def extract_datetime(lines):
    for line in lines:
        # Look for the line containing LabChart timestamp info
        if "ExcelDateTime" in line:

            # Use regular expression to extract date and time
            match = re.search(r"(\d{2}/\d{2}/\d{4})\s+(\d{1,2}:\d{2}:\d{2})", line)

            if match:
                # Extract date and time components
                date = match.group(1).replace("/", "-")  # format: DD-MM-YYYY
                time = match.group(2).replace(":", "-")  # format: HH-MM-SS

                # Return combined timestamp for file naming
                return f"{date}_{time}"

    # If no timestamp is found, return default name
    return "unknown_time"


# ---------------------------------------------------------
# Function: clean_labchart_file
# Purpose:
#   Clean LabChart exported .txt file for Kubios HRV analysis:
#   - Remove metadata lines
#   - Convert decimal commas to dots
#   - Separate data from metadata
#   - Save clean file + metadata file
# ---------------------------------------------------------
def clean_labchart_file(input_file):

    # Get path to user's Desktop (cross-platform compatible)
    desktop = os.path.join(os.path.expanduser("~"), "Desktop")

    # Create output folder on Desktop
    output_dir = os.path.join(desktop, "LabChart_clean")
    os.makedirs(output_dir, exist_ok=True)

    # Open input file safely
    # strip() removes spaces and quotes around file path
    with open(input_file.strip().strip('"'), "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    # Extract recording date and time for file naming
    timestamp = extract_datetime(lines)

    # Lists to store processed data
    cleaned_lines = []    # numeric EEG/ECG data
    metadata_lines = []   # LabChart metadata

    # -----------------------------------------------------
    # Loop through each line of the file
    # -----------------------------------------------------
    for line in lines:
        line = line.strip()  # remove spaces and line breaks

        # If line contains "=", it is metadata → store separately
        if "=" in line:
            metadata_lines.append(line)
            continue  # skip to next line

        # Skip empty lines
        if line == "":
            continue

        # Replace decimal commas with decimal points
        line = line.replace(",", ".")

        # Keep only lines that contain numbers
        if any(char.isdigit() for char in line):
            cleaned_lines.append(line)

    # -----------------------------------------------------
    # Create output file names
    # -----------------------------------------------------
    base_name = f"ECG_{timestamp}"

    data_file = os.path.join(output_dir, base_name + "_kubios.txt")
    meta_file = os.path.join(output_dir, base_name + "_metadata.txt")

    # -----------------------------------------------------
    # Write cleaned data file (Kubios-ready)
    # -----------------------------------------------------
    with open(data_file, "w", encoding="utf-8") as f:
        for line in cleaned_lines:
            f.write(line + "\n")

    # -----------------------------------------------------
    # Write metadata file (for traceability)
    # -----------------------------------------------------
    with open(meta_file, "w", encoding="utf-8") as f:
        for line in metadata_lines:
            f.write(line + "\n")

    # -----------------------------------------------------
    # Print output paths for user confirmation
    # -----------------------------------------------------
    print("\nProcessing complete!")
    print("Files created:")
    print("Data file:", data_file)
    print("Metadata file:", meta_file)


# ---------------------------------------------------------
# Main program entry point
# ---------------------------------------------------------
if __name__ == "__main__":

    # Ask user to provide path to LabChart file
    file_path = input("Enter LabChart .txt file path: ")

    # Run cleaning function
    clean_labchart_file(file_path)

Enter LabChart .txt file path:  C:\Users\judupont\Downloads\ecg_labchart.txt



Processing complete!
Files created:
Data file: C:\Users\judupont\Desktop\LabChart_clean\ECG_29-04-2026_9-24-41_kubios.txt
Metadata file: C:\Users\judupont\Desktop\LabChart_clean\ECG_29-04-2026_9-24-41_metadata.txt
